In [1]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer

import re
from bs4 import BeautifulSoup

from spacy.lang.en.stop_words import STOP_WORDS
import en_core_web_sm

from pathlib import Path
from scipy.sparse import save_npz
import joblib


1. Importer

In [2]:
data = pd.read_csv('/Users/nicolasbour/Desktop/Projets_IA/Projets_certif/bloc_03_Machine_Learning/03_the_north_face/Nico/data/processed/sample_clean.csv')

In [3]:
pd.set_option('display.max_colwidth', None)
display(data.head(1))

,id,description_texte,details,fabric,weight,pays
0,1,"active classic boxers - s reason boxers cult favorite - cool , especially sticky situations . quick - drying , lightweight underwear takes minimal space travel pack . exposed , brushed waistband offers - - skin softness , - panel construction traditional boxer classic fit , functional fly . 3.7 - oz 100 % recycled polyester moisture - wicking performance . inseam size m 4 1/2 . recyclable common threads recycling program .","silky capilene 1 fabric ultralight , breathable quick - - dry exposed , brushed elastic waistband comfort 5 - panel construction traditional boxer inseam size m 4 1/2",3.7 - oz 100 % - recycled polyester gladiodor natural odor control garment . recyclable common threads recycling program,99 g 3.5 oz,Mexico


In [4]:
# sample = data.sample(n=5, random_state=42)
data = data.copy()

In [5]:
data.shape

(500, 6)

2. tokenizer

In [6]:
nlp = en_core_web_sm.load()


data["description_tokenized"] = data["description_texte"].apply(
    lambda texte: " ".join(
        token.text for token in nlp(texte) if not token.is_stop
    ) if pd.notna(texte) else texte
)

In [7]:
data.head(1)

,id,description_texte,details,fabric,weight,pays,description_tokenized
0,1,"active classic boxers - s reason boxers cult favorite - cool , especially sticky situations . quick - drying , lightweight underwear takes minimal space travel pack . exposed , brushed waistband offers - - skin softness , - panel construction traditional boxer classic fit , functional fly . 3.7 - oz 100 % recycled polyester moisture - wicking performance . inseam size m 4 1/2 . recyclable common threads recycling program .","silky capilene 1 fabric ultralight , breathable quick - - dry exposed , brushed elastic waistband comfort 5 - panel construction traditional boxer inseam size m 4 1/2",3.7 - oz 100 % - recycled polyester gladiodor natural odor control garment . recyclable common threads recycling program,99 g 3.5 oz,Mexico,"active classic boxers - s reason boxers cult favorite - cool , especially sticky situations . quick - drying , lightweight underwear takes minimal space travel pack . exposed , brushed waistband offers - - skin softness , - panel construction traditional boxer classic fit , functional fly . 3.7 - oz 100 % recycled polyester moisture - wicking performance . inseam size m 4 1/2 . recyclable common threads recycling program ."


In [8]:
docs = []
for texte in data["description_texte"].dropna():
    doc = nlp(texte)
    print("Tokens:", [token.text for token in doc])
    print("Lemma:", [token.lemma_ for token in doc])
    print("Indices:", [token.i for token in doc])
    print("Alpha Numeric:", [token.is_alpha for token in doc])
    print("Numbers:", [token.like_num for token in doc])
    print("Stopwords restants:", [token.text for token in doc if token.is_stop])
    docs.append(doc)

Tokens: ['active', 'classic', 'boxers', '-', 's', 'reason', 'boxers', 'cult', 'favorite', '-', 'cool', ',', 'especially', 'sticky', 'situations', '.', 'quick', '-', 'drying', ',', 'lightweight', 'underwear', 'takes', 'minimal', 'space', 'travel', 'pack', '.', 'exposed', ',', 'brushed', 'waistband', 'offers', '-', '-', 'skin', 'softness', ',', '-', 'panel', 'construction', 'traditional', 'boxer', 'classic', 'fit', ',', 'functional', 'fly', '.', '3.7', '-', 'oz', '100', '%', 'recycled', 'polyester', 'moisture', '-', 'wicking', 'performance', '.', 'inseam', 'size', 'm', '4', '1/2', '.', 'recyclable', 'common', 'threads', 'recycling', 'program', '.']
Lemma: ['active', 'classic', 'boxer', '-', 's', 'reason', 'boxer', 'cult', 'favorite', '-', 'cool', ',', 'especially', 'sticky', 'situation', '.', 'quick', '-', 'dry', ',', 'lightweight', 'underwear', 'take', 'minimal', 'space', 'travel', 'pack', '.', 'expose', ',', 'brush', 'waistband', 'offer', '-', '-', 'skin', 'softness', ',', '-', 'panel'

In [9]:
docs

[active classic boxers - s reason boxers cult favorite - cool , especially sticky situations . quick - drying , lightweight underwear takes minimal space travel pack . exposed , brushed waistband offers - - skin softness , - panel construction traditional boxer classic fit , functional fly . 3.7 - oz 100 % recycled polyester moisture - wicking performance . inseam size m 4 1/2 . recyclable common threads recycling program .,
 active sport boxer briefs - skinning glory requires movement boxers deciding poach route . form - fitting active sport boxer briefs breathable 93 % polyester 71 % recycled fabric s fast - wicking , dries quickly 7 % spandex stretch seamless waistband soft leg edges won t roll bind . gusseted , flat - sewn 6 inseam size m offset prevent inner - thigh chafe . fly - free smooth panel . recyclable common threads recycling program .,
 active sport briefs - superbreathable - fly briefs minimalist s choice high - octane endeavors . blend fast - wicking , quick - drying 9

In [10]:
print(STOP_WORDS)

{'was', 'almost', 'besides', 'top', 'using', '‘m', '‘s', 'bottom', 'over', 'few', 'much', 'nine', '’ve', 'must', '’re', 'take', 'three', 'through', 'whose', 'such', 'own', 'be', 'against', 'did', 'could', 'or', 'at', "'d", 'make', 'whom', 'would', 'had', 'thence', 'yourself', 'become', 'seemed', "'s", 'themselves', 'two', 'herein', 'been', 'that', 'hereby', 'beside', 'some', 'below', 'any', 'around', '‘d', 'them', 'herself', 'name', 'ours', 'than', 'please', 'towards', 'a', "n't", 'last', 'under', 'same', 'everything', 'though', 'made', 'whenever', 'back', 'down', 'other', 'another', 'anywhere', 'each', 'and', 'therein', 'beforehand', 'well', 'being', 'various', 'enough', 'mostly', 'put', 'our', 'former', 'whereas', 'whereafter', 'thereafter', 'with', 'very', 'call', 'hereafter', 'rather', 'every', 'always', 'part', 'are', 'until', 'show', 'otherwise', 'she', 'really', 'ever', 'out', 'us', 'n‘t', 'doing', 'i', 'all', 'full', '’s', 'have', 'from', 'together', 'both', 'beyond', 'sometime

3. Lemmatiser

In [11]:
textes = data["description_texte"].fillna("").astype(str)

data["description_lemmatized"] = [
    " ".join(
        token.lemma_.lower()
        for token in doc
        if not token.is_stop and token.is_alpha
    )
    for doc in nlp.pipe(textes, batch_size=100)
]

pd.set_option('display.max_colwidth', None)
data[["description_texte", "description_tokenized", "description_lemmatized"]].head(1)

,description_texte,description_tokenized,description_lemmatized
0,"active classic boxers - s reason boxers cult favorite - cool , especially sticky situations . quick - drying , lightweight underwear takes minimal space travel pack . exposed , brushed waistband offers - - skin softness , - panel construction traditional boxer classic fit , functional fly . 3.7 - oz 100 % recycled polyester moisture - wicking performance . inseam size m 4 1/2 . recyclable common threads recycling program .","active classic boxers - s reason boxers cult favorite - cool , especially sticky situations . quick - drying , lightweight underwear takes minimal space travel pack . exposed , brushed waistband offers - - skin softness , - panel construction traditional boxer classic fit , functional fly . 3.7 - oz 100 % recycled polyester moisture - wicking performance . inseam size m 4 1/2 . recyclable common threads recycling program .",active classic boxer s reason boxer cult favorite cool especially sticky situation quick dry lightweight underwear take minimal space travel pack expose brush waistband offer skin softness panel construction traditional boxer classic fit functional fly oz recycled polyester moisture wicke performance inseam size m recyclable common thread recycle program


4. Vectoriser

In [12]:
vectorizer = TfidfVectorizer(
    stop_words="english",
    max_df=0.95,
    min_df=2
)

# Remplace les valeurs manquantes par une chaîne vide
matrice_tfidf = vectorizer.fit_transform(data["description_lemmatized"].fillna(""))

print("Dimensions :", matrice_tfidf.shape)
print("Exemples de mots :", vectorizer.get_feature_names_out()[:20])

Dimensions : (500, 1865)
Exemples de mots : ['ability' 'abrasion' 'absorb' 'abuse' 'abusive' 'access' 'accessory'
 'accommodate' 'accomplish' 'accord' 'action' 'active' 'activity' 'adapt'
 'adaptability' 'adaptable' 'add' 'addition' 'additional' 'adhere']


5. exporter

In [13]:
processed_dir = Path("../data/processed")
models_dir = Path("../models")

processed_dir.mkdir(parents=True, exist_ok=True)
models_dir.mkdir(parents=True, exist_ok=True)

# Matrice TF-IDF
save_npz(processed_dir / "tfidf_matrix.npz", matrice_tfidf)

# Vectoriseur déjà entraîné
joblib.dump(vectorizer, models_dir / "tfidf_vectorizer.joblib")

# Correspondance : ligne de la matrice → produit
data[["id"]].to_csv(
    processed_dir / "tfidf_row_ids.csv",
    index=False,
)